# time-stage-instrumentation — worked example 1: Time two stages with perf_counter

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `time-stage-instrumentation`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

To profile a loop, wrap each named stage between two `time.perf_counter()` reads and add the difference into a per-stage accumulator. `perf_counter` is a monotonic high-resolution clock — the right tool for measuring durations, unlike `time.time()` which can jump if the wall clock is adjusted.

## Worked solution

We want the total seconds spent in a 'compute' stage versus a 'sync' stage across several iterations.

1. We initialize an accumulator dict with both stage names at `0.0`.
2. For each iteration, we record `t0 = time.perf_counter()` immediately before the work, run the work, then add `time.perf_counter() - t0` to that stage's running total.
3. We use `perf_counter` (not `time.time()`) at both endpoints so the delta is a precise elapsed-time measurement.
4. After the loop, the accumulator holds the summed seconds per stage.

Because the 'compute' sleep is longer than the 'sync' sleep, the printed compute total is the larger of the two — exactly what a profiler would show when compute dominates.

In [ ]:
import time

def profile_two_stages(n_iters, sleep_compute, sleep_sync):
    stages = {'compute': 0.0, 'sync': 0.0}
    for _ in range(n_iters):
        t0 = time.perf_counter()
        time.sleep(sleep_compute)
        stages['compute'] += time.perf_counter() - t0

        t0 = time.perf_counter()
        time.sleep(sleep_sync)
        stages['sync'] += time.perf_counter() - t0
    return stages

stages = profile_two_stages(3, 0.01, 0.002)
print('compute:', round(stages['compute'], 4))
print('sync:', round(stages['sync'], 4))
print('compute dominates:', stages['compute'] > stages['sync'])